# Generacion de dataset

In [1]:
import pandas as pd
import numpy as np
import timeit

print("Generando dataset de 1,000,000 de filas...")
n = 1_000_000
df = pd.DataFrame({
    'lat1': np.random.uniform(-90, 90, n),
    'lon1': np.random.uniform(-180, 180, n),
    'lat2': np.random.uniform(-90, 90, n),
    'lon2': np.random.uniform(-180, 180, n)
})

# Función de apoyo para el cálculo (Distancia Euclidiana simple)
def haversine_python(lat1, lon1, lat2, lon2):
    return ((lat2 - lat1)**2 + (lon2 - lon1)**2)**0.5

print("Dataset listo. Comienza el benchmark.\n")

Generando dataset de 1,000,000 de filas...
Dataset listo. Comienza el benchmark.



## Iteracion por for loop

Este proceso es lento para procesar los datos debido a que Python es un interprete y al usar el for in no esta aprovechando el beneficio de Pandas que el Block Manager distribuye en columnas los elementos ordenados uno al lado del otro, al hacer el for in, por cada fila, python salta de un bloque a otro para envolver el resultado en un objeto, lo que acaba provocando cache miss constante y la CPU queda esperando a la RAM.

In [2]:
%%timeit
distancias = []
for i in range(len(df)):
    d = haversine_python(df.iloc[i]['lat1'], df.iloc[i]['lon1'], 
                         df.iloc[i]['lat2'], df.iloc[i]['lon2'])
    distancias.append(d)
df['distancia_loop'] = distancias

43.3 s ± 2.11 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
print(df[['lat1', 'lon1', 'lat2', 'lon2', 'distancia_loop']].head(10))

        lat1        lon1       lat2        lon2  distancia_loop
0  65.630758   61.506792   4.739178 -153.134685      223.111515
1 -10.363340  -69.555377 -12.612604   48.082689      117.659568
2 -78.858295  -44.503215 -84.637914  122.442763      167.045993
3 -50.754922  -90.671090  69.070160   95.958707      221.785327
4  43.558242    7.889517  58.295765 -152.391792      160.957425
5 -66.513745   40.876256  79.691942   62.948522      147.862395
6  46.232404 -118.745106  81.940629  -21.210240      103.865910
7 -59.131301  159.956581  21.334798  101.117273       99.683786
8 -12.744148   -6.688517 -24.857590  -98.594171       92.700510
9  65.452163   49.461128   9.287511  -10.762256       82.348795


In [ ]:
%%timeit
df['distancia_apply'] = df.apply(lambda row: haversine_python(row['lat1'], row['lon1'], row['lat2'], row['lon2']), axis=1)

In [7]:
%%timeit
df['distancia_vector'] = ((df['lat2'] - df['lat1'])**2 + (df['lon2'] - df['lon1'])**2)**0.5

9.03 ms ± 52.4 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [11]:
# Extraemos los arrays de C puros (esto no copia datos, solo apunta a ellos)
lat1 = df['lat1'].values
lon1 = df['lon1'].values
lat2 = df['lat2'].values
lon2 = df['lon2'].values

# Usamos las funciones matemáticas de NumPy directamente
# %%timeit
distancia_pro = np.sqrt(np.square(lat2 - lat1) + np.square(lon2 - lon1))

# Si quieres guardarlo de vuelta en el DataFrame:
df['distancia_numpy'] = distancia_pro